# Day 04 — NGS pathway delta P (n=64)

**Slides (tải về):** [`day04_slides.pptx`](_static/slides/day04_slides.pptx)

Mục tiêu:
- Lấy xác suất dự đoán p(EGFR+) từ mô hình.
- Ghép với bảng pathway (n=64).
- Tính delta mean P và delta median P giữa nhóm mutated và wildtype.
- Vẽ biểu đồ và viết diễn giải ngắn.

Lưu ý: file pathway ở đây là demo để học quy trình.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (6,4)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Nếu chạy trên Colab và không thấy file, notebook sẽ tải demo CSV từ GitHub.
GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO_NAME = "YOUR_REPO_NAME"
BRANCH = "main"

def download_from_github(rel_path: str) -> Path:
    import urllib.request
    url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{rel_path}"
    out_path = Path(rel_path).name
    print("Tải file:", url)
    urllib.request.urlretrieve(url, out_path)
    return Path(out_path)

def load_csv(rel_path: str, fallback_upload: bool = True) -> pd.DataFrame:
    candidates = [
        Path(rel_path),
        Path("../")/rel_path,
        Path("../../")/rel_path,
        Path("data")/Path(rel_path).name,
        Path("_static/data")/Path(rel_path).name,
        Path("../data")/Path(rel_path).name,
    ]
    for p in candidates:
        if p.exists():
            return pd.read_csv(p)

    # Thử tải từ GitHub
    try:
        p = download_from_github(rel_path)
        return pd.read_csv(p)
    except Exception as e:
        print("Không tải được từ GitHub:", e)

    # Cuối cùng: upload thủ công
    if fallback_upload:
        try:
            from google.colab import files  # type: ignore
            uploaded = files.upload()
            if len(uploaded) > 0:
                up_path = Path(list(uploaded.keys())[0])
                return pd.read_csv(up_path)
        except Exception:
            pass

    raise FileNotFoundError("Không tìm thấy CSV. Cần clone repo hoặc upload file.")

df = load_csv("data/nsclc_egfr_radiomics_simplified.csv")
df.head()


In [ ]:
# Load NGS pathway demo (n=64)
ngs = load_csv("data/ngs_pathway_demo_64.csv")
ngs.head(), ngs.shape


## 1) Train mô hình trên cohort 200 và lấy predicted probability

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

y = df["egfr_mutation"].astype(int)

# dùng ring1 để minh hoạ
feature_set = "ring1"
cols = [c for c in df.columns if c.startswith(feature_set + "_")]
X = df[cols].copy()

pre = ColumnTransformer([("num", StandardScaler(), cols)], remainder="drop")
model = LogisticRegression(max_iter=2000)
pipe = Pipeline([("pre", pre), ("model", model)])

pipe.fit(X, y)

df_pred = df[["patient_id"]].copy()
df_pred["p_pred"] = pipe.predict_proba(X)[:,1]
df_pred.head()


## 2) Ghép predicted probability với pathway

In [ ]:
m = ngs.merge(df_pred, on="patient_id", how="left")
m.head()


## 3) Tính delta mean và delta median theo pathway

In [ ]:
from scipy.stats import mannwhitneyu

pathway_cols = [c for c in m.columns if c.startswith("pathway_")]
rows = []

for p in pathway_cols:
    group1 = m.loc[m[p]==1, "p_pred"].dropna()
    group0 = m.loc[m[p]==0, "p_pred"].dropna()

    delta_mean = group1.mean() - group0.mean()
    delta_median = group1.median() - group0.median()

    # Mann-Whitney U test (2-sided)
    if len(group1) >= 5 and len(group0) >= 5:
        stat, pval = mannwhitneyu(group1, group0, alternative="two-sided")
    else:
        pval = np.nan

    rows.append({
        "pathway": p,
        "n_mutated": len(group1),
        "n_wildtype": len(group0),
        "mean_mutated": group1.mean(),
        "mean_wildtype": group0.mean(),
        "delta_mean": delta_mean,
        "median_mutated": group1.median(),
        "median_wildtype": group0.median(),
        "delta_median": delta_median,
        "p_value": pval,
    })

res = pd.DataFrame(rows).sort_values("delta_median", ascending=False)
res


## 4) Vẽ biểu đồ delta median P

In [ ]:
plt.bar(res["pathway"], res["delta_median"])
plt.xticks(rotation=45, ha="right")
plt.axhline(0, linestyle="--")
plt.title("Delta median predicted probability by pathway")
plt.ylabel("delta median P (mutated - WT)")
plt.tight_layout()
plt.show()


## 5) Boxplot minh hoạ cho 1 pathway

In [ ]:
# chọn pathway đứng đầu bảng
p0 = res.iloc[0]["pathway"]
g1 = m.loc[m[p0]==1, "p_pred"].dropna()
g0 = m.loc[m[p0]==0, "p_pred"].dropna()

plt.boxplot([g0, g1], labels=["WT", "Mutated"])
plt.title(p0)
plt.ylabel("p_pred")
plt.show()


## Bài tập cuối buổi
1) Pathway nào có delta median P lớn nhất? Delta là bao nhiêu?
2) Vì sao cần báo cáo n_mutated và n_wildtype?
3) Vì sao dùng median hợp lý khi n nhỏ?
4) Viết 5 dòng diễn giải: kết quả gợi ý gì và hạn chế lớn nhất là gì.